# 01 · Datos e HDFS

O dataset empregado é [Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows): un CSV con 12 columnas. Ten trampas reais: comas dentro de campos entre comiñas (reparto, países, xéneros), datas en texto (`September 25, 2021`) e duracións mixtas (`90 min`, `2 Seasons`).

O CSV está xa dentro de: `datos/netflix_titles.csv` e térémolo que copiar ao HDFS para traballar con el.

As variables e as funcións (sh, hdfs...) están definidas en `utils_hive.py`

In [2]:
from utils_hive import *

Comprobamos a validez do arquivo:

In [4]:
CSV

'/home/xunta/edu/p037/notebook/exercicio_hive_netflix/datos/netflix_titles.csv'

In [3]:
sh("head -n 3 " + CSV)
sh("wc -l " + CSV)

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
s2,TV Show,Blood & Water,,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."

8810 /home/xunta/edu/p037/notebook/exerc

0

Cargamos variables de contorno para que todo funcione:

In [7]:
hdfs('-ls')
sh("hostname; which hdfs; echo $HADOOP_CONF_DIR")
print("HOME no HDFS: " + HDFS_HOME)

Found 5 items
drwx------   - xuedup037 xunta          0 2026-11-27 12:00 .Trash
drwxr-xr-x   - xuedup037 xunta          0 2026-11-26 21:34 .sparkStaging
drwx------   - xuedup037 xunta          0 2026-11-27 13:11 .staging
drwx------   - xuedup037 xunta          0 2026-11-27 12:55 datasets
drwx------   - xuedup037 xunta          0 2026-11-27 13:01 netflix

cdh61-login3.bd.cluster.cesga.es
/usr/bin/hdfs
/etc/hadoop/conf

HOME no HDFS: /user/xuedup037


## Subindo os datos ao HDFS:

Agora copiamos o CSV do HOME ao HDFS. Vai só nun directorio porque a `LOCATION` dunha táboa Hive é un **directorio**: Hive le todos os ficheiros que conteña.

**Opción 1**: Mediante comandos
```
hdfs dfs -mkdir -p datasets/
hdfs dfs -put datos/netflix_titles.csv datasets/
dfs dfs -checksum datasets/netflix_titles.csv
```

O checksum debería dar:

`netflix_titles.csv	MD5-of-0MD5-of-512CRC32C	000002000000000000000000a56e391d0a5f272177785b9f6ca49086`

**Opción 2**: Mediante o código seguinte (executa só se non o subiches a man):

In [4]:
HDFS_DATASETS = HDFS_HOME + "/datasets"

hdfs("-mkdir -p " + HDFS_DATASETS)
hdfs("-put -f " + CSV + " " + HDFS_DATASETS + "/")
hdfs("-chmod -R go-rwx " + HDFS_DATASETS)      # só ti

0

Cantos bloques ocupa o ficheiro e con cantas réplicas ten?

In [5]:
hdfs('-stat "réplicas=%r bloque=%o bytes tamaño=%b bytes" ' + HDFS_DATASETS + "/netflix_titles.csv")

réplicas=3 bloque=134217728 bytes tamaño=3399671 bytes



0

Comprobamos o checksum do arquivo que debería dar:

`netflix_titles.csv	MD5-of-0MD5-of-512CRC32C	000002000000000000000000a56e391d0a5f272177785b9f6ca49086`

In [6]:
hdfs('-checksum ' + HDFS_DATASETS + "/netflix_titles.csv")

/user/xuedup037/datasets/netflix_titles.csv	MD5-of-0MD5-of-512CRC32C	000002000000000000000000a56e391d0a5f272177785b9f6ca49086



0